In [24]:
using LowLevelFEM, LinearAlgebra, IterativeSolvers

openGeometry("I-szelveny-2.geo")
#openGeometry("I-szelveny-transfinite.geo")

In [25]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u)
U.non * U.pdim

40338

In [26]:
@time K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U), Ω="body")

  0.852815 seconds (122.29 k allocations: 104.285 MiB, 25.29% gc time)


sparse([1, 2, 3, 115, 116, 117, 544, 545, 546, 547  …  35478, 35953, 35954, 35955, 38227, 38228, 38229, 40336, 40337, 40338], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  40338, 40338, 40338, 40338, 40338, 40338, 40338, 40338, 40338, 40338], [278292.566803007, 97081.80773573398, 153385.20528700302, -26741.733824488707, 13105.80636129082, 2619.580282242783, 45968.090423427646, -1399.5669920043165, 38671.82341937133, 25110.684671834082  …  -330721.93311894656, 124582.65746023381, 23158.34748816608, -69476.43579369927, -77030.21660485685, 8169.756825662913, -205899.9822406297, -51408.36686734077, 260436.17385856068, 2.8283609109073346e6], 40338, 40338)

In [27]:
φ = 0.01     # total twist angle [rad]

bc_left = BoundaryCondition(
    "left",
    ux = 0.0,
    uy = 0.0
)

bc_right = BoundaryCondition(
    "right",
    ux = (x,y,z) -> -φ*y,
    uy = (x,y,z) ->  φ*x
)

BoundaryCondition("right", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => var"#17#18"(), :ux => var"#15#16"()))

In [28]:
bc_anchor = BoundaryCondition("A", uz=0)

BoundaryCondition("A", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uz => 0))

In [29]:
f = ∫(U ⋅ [0, 0, 0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [30]:
using IterativeSolvers, Preconditioners
@time u = solveField(Symmetric(K), f, support=[bc_left, bc_right, bc_anchor]);

  1.705994 seconds (1.13 M allocations: 348.209 MiB, 19.52% gc time, 31.14% compilation time)


In [31]:
showDoFResults(u, :uvec, name="displacement", factor=1);

In [32]:
#u1 = nodesToElements(u, onPhysicalGroup="mid")
ε = (u ∘ ∇ + ∇ ∘ u) / 2

E = mat.E
ν = mat.ν
I = unitTensor(ε)

σ = E / (1+ν) * (ε + ν / (1 - 2ν) * trace(ε) * I);

In [33]:
n = normalVector(U, "mid")

σ = elementsToElements(σ, fromPhysicalGroup="bodyL", onPhysicalGroup="mid")

τ = σ * n
τ -= (τ ⋅ n) * n

τ = elementsToNodes(τ)
τ = nodesToElements(τ, onPhysicalGroup="mid")

elementwise VectorField
[[111.23835495014367; -116.94722847972818; … ; -128.5628927964729; 0.0;;], [66.56594444491873; -193.71665955565396; … ; -128.5628927964729; 0.0;;], [192.5835277058574; 78.94551275160876; … ; 123.72740843804638; 0.0;;], [116.4551732752428; 130.06422015481664; … ; 123.72740843804638; 0.0;;], [-110.33467002712499; -120.2999099546789; … ; -129.44137258777977; 0.0;;], [-194.61204908236067; -81.76475421821692; … ; -129.44137258777977; 0.0;;], [-388.05561706001293; -372.3668559912478; … ; -157.45505994683933; 0.0;;], [-116.71610814244902; -619.9375431342285; … ; -455.7573201481599; 0.0;;], [-171.21865491050193; -455.7573201481599; … ; -281.9227653538987; 0.0;;], [-231.97331683441763; -281.9227653538987; … ; -225.22769855894992; 0.0;;]  …  [264.8789687986466; 0.7683892418867068; … ; -10.56105097158567; 0.0;;], [230.79424963528356; 9.309573974012583; … ; 6.841860248376119; 0.0;;], [-215.95085510197774; -3.8485905473962685; … ; -11.812843789075048; 0.0;;], [-267.595962698

In [34]:
R = nodePositionVector(U)
R = nodesToElements(R, onPhysicalGroup="mid")
∫(U, "mid", (R × τ) ⋅ n)

6.537865166262463e6

In [35]:
showElementResults(τ, name="shear stress");

In [36]:
showDoFResults(n, name="normal vector");

In [37]:
openPostProcessor()